In [2]:
from langchain.memory import ConversationBufferMemory, ChatMessageHistory


In [3]:
memory = ConversationBufferMemory()
memory.chat_memory.add_user_message("Hello, my name is Jay.")
memory.chat_memory.add_ai_message("What's up?")


C:\Users\Administrator\AppData\Local\Temp\ipykernel_16912\1548766737.py:1: LangChainDeprecationWarning: Please see the migration guide at: https://python.langchain.com/docs/versions/migrating_memory/
  memory = ConversationBufferMemory()


In [6]:
from langchain.chains import ConversationChain
from langchain_ollama import ChatOllama
from langchain.callbacks.manager import CallbackManagerForLLMRun
from langchain_community.llms import Ollama

ollama = Ollama(model="llama2:latest")


query = "Tell me a joke"

for chunks in ollama.stream(query):
    print(chunks)




C:\Users\Administrator\AppData\Local\Temp\ipykernel_16912\174869729.py:6: LangChainDeprecationWarning: The class `Ollama` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the :class:`~langchain-ollama package and should be used instead. To use it run `pip install -U :class:`~langchain-ollama` and import as `from :class:`~langchain_ollama import OllamaLLM``.
  ollama = Ollama(model="llama2:latest")




S
ure
,
 here
'
s
 one
:




Why
 don
'
t
 scient
ists
 trust
 atoms
?


B
ecause
 they
 make
 up
 everything
!




I
 hope
 you
 found
 that
 am
using
!
 Do
 you
 want
 to
 hear
 another
 one
?



In [7]:
import os
import json

from langchain_community.llms import Ollama

from flask import Flask, Response, request
from flask import stream_with_context

app = Flask(__name__)

OLLAMA_BASE_URL = os.getenv("OLLAMA_BASE_URL", "http://0.0.0.0:11434")
OLLAMA_MODEL = os.getenv("OLLAMA_MODEL", "llama2:latest")

llm = Ollama(model=OLLAMA_MODEL, base_url=OLLAMA_BASE_URL)

def generate_tokens(question):
    for chunks in llm.stream(question):
        yield chunks

@app.route("/users/chat", methods=["POST"])
def ask_ai():
    def generate_json(question):
        with app.app_context():  # Ensure we're within the application context
            full_content = ""
            for token in generate_tokens(question):
                full_content += token
                json_data = {
                    "model": OLLAMA_MODEL,
                    "content": token,
                    "done": False
                }
                json_str = json.dumps(json_data)  # Convert JSON data to a string
                json_bytes = json_str.encode('utf-8')  # Encode JSON string to bytes
                yield json_bytes
                yield b'\n'  # Yield newline as bytes

            # Once streaming is finished, yield one last JSON object with "done" set to True
            json_data = {
                "model": OLLAMA_MODEL,
                "full_content": full_content,
                "done": True
            }
            json_str = json.dumps(json_data)  # Convert JSON data to a string
            json_bytes = json_str.encode('utf-8')  # Encode JSON string to bytes
            yield json_bytes
    
    request_data = request.json
    question = request_data.get("question")
    return Response(stream_with_context(generate_json(question)), mimetype='application/json')


if __name__ == '__main__':
    app.run(debug=True)


 * Serving Flask app '__main__'
 * Debug mode: on


 * Running on http://127.0.0.1:5000
Press CTRL+C to quit
 * Restarting with stat


SystemExit: 1

c:\Users\Administrator\Documents\Repos\CrimeProject\.venv\Lib\site-packages\IPython\core\interactiveshell.py:3675: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)
